In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

In [ ]:
class Mission5(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=40)
        # シミュレータでは「最後の標識付近 → 最初の標識付近」の順に通過したら成功とする
        self.goals = [
            GoalCircle((4.2, 0.7), 0.3, should_stop=False),
            GoalCircle((2.7, -0.1), 0.3, should_stop=False),
        ]
        self.initial_xy = (1.7, 0.0)
        self.random_d_xy = (0.1, 0.1)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=2.7, y=-0.1, name="original"),
                Sign(x=3.5, y=0.1, name="original"),
                Sign(x=4.2, y=0.7, name="original"),
            ]
        )

    @staticmethod
    def command_func(alive, *, move, rotate, wait, search, **kwargs):
        """回答例：標識に向かって進み、標識が見つからなくなったらUターンする"""
        while alive():
            pos = search(name="original")
            if pos is None:
                ######## ここから下にプログラムを書こう
                rotate(w=90, t=2)
                wait()
                ######## ここより上にプログラムを書こう
            else:
                if pos.theta > 5:
                    rotate(w=45)
                elif pos.theta < -5:
                    rotate(w=-45)
                else:
                    move(v=0.2)


sim = CarSim(prop, Mission5())
sim.run()
SimDrawer(sim).show()

In [ ]:
# 複数回実行して必ず成功するかのチェック
CHECK_CNT = 10
ok = 0
for i in range(CHECK_CNT):
    mission = Mission5()
    sim = CarSim(prop, mission)
    complete = sim.run()
    if complete:
        goal_cnt = sim.history.goal_cnt[-1]
        goals = len(mission.goals)
        goaled = goal_cnt == goals
        if goaled:
            ok += 1
        else:
            SimDrawer(sim).show()
            break
        print("-------------------------------OK:", ok)
